# Transformer & T5 — Demo các concept trong Chương 2

Notebook này minh hoạ **8 concept** dùng trong hệ thống Spell-Checker:

| Mục | Concept | Cell |
|---|---|---|
| 2.2.1 | Sequence-to-Sequence (Seq2Seq) | 1 |
| 2.2.2 | Kiến trúc Transformer (encoder block) | 2 |
| 2.1.4 | Self-Attention: `softmax(QKᵀ/√d_k)V` | 3 |
| 2.1.5 | Multi-Head Attention | 4 |
| 2.1.6 | Position-wise Feed-Forward Networks | 5 |
| 2.1.7 | Positional Encoding (sinusoidal) | 6 |
| 2.1.8 | Residual Connection + Layer Normalization | 7 |
| 2.2.3 | Mô hình T5 (Text-to-Text Transfer Transformer) | 8 |

**Cách dùng**: Run all → screenshot output cho báo cáo Chương 2.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')

## 2.2.1. Sequence-to-Sequence (Seq2Seq)

**Kiến trúc Encoder-Decoder**: encode chuỗi input → context vector → decode thành chuỗi output.

Áp dụng cho task của ta: **input = câu sai, output = câu đúng**.

In [ ]:
class Seq2Seq(nn.Module):
    """Seq2Seq cơ bản với LSTM — dùng minh hoạ cho Chương 2."""
    def __init__(self, vocab_size, hidden_dim=256):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, hidden_dim)
        self.encoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc      = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, tgt):
        # ENCODER: input → context
        src_emb = self.embed(src)
        _, (h, c) = self.encoder(src_emb)         # h, c là context
        # DECODER: context → output
        tgt_emb = self.embed(tgt)
        out, _ = self.decoder(tgt_emb, (h, c))
        return self.fc(out)

model = Seq2Seq(vocab_size=1000)
src = torch.randint(0, 1000, (2, 10))   # batch=2, len=10
tgt = torch.randint(0, 1000, (2, 10))
out = model(src, tgt)
print(f'Input shape:  {src.shape}        # (batch, seq_len)')
print(f'Output shape: {out.shape}  # (batch, seq_len, vocab_size)')
print(f'Tham số:      {sum(p.numel() for p in model.parameters()):,}')

## 2.1.4. Self-Attention (Scaled Dot-Product Attention)

Công thức:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Ý nghĩa**: tính độ "chú ý" giữa mọi cặp token trong câu.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Implement công thức Attention(Q,K,V) = softmax(QKᵀ/√d_k)V"""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)   # QK^T / sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention = F.softmax(scores, dim=-1)                          # softmax
    output = torch.matmul(attention, V)                            # x V
    return output, attention

# Demo: câu 5 token, d_model=64
seq_len, d_model = 5, 64
Q = torch.randn(1, seq_len, d_model)
K = torch.randn(1, seq_len, d_model)
V = torch.randn(1, seq_len, d_model)

output, attention = scaled_dot_product_attention(Q, K, V)
print(f'Q, K, V shape: {Q.shape}                    # (batch, seq_len, d_k)')
print(f'Attention shape: {attention.shape}             # (batch, seq_len, seq_len) — ma trận chú ý')
print(f'Output shape:    {output.shape}\n')

# Visualize attention matrix
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attention[0].detach().numpy(), cmap='viridis')
ax.set_xlabel('Key tokens'); ax.set_ylabel('Query tokens')
ax.set_title('Self-Attention Matrix\n(softmax(QKᵀ/√d_k))')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attention[0, i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 2.1.5. Multi-Head Attention

Chạy SONG SONG nhiều "head" attention, mỗi head học một kiểu quan hệ khác nhau:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

Trong T5-base: `num_heads = 12`, mỗi head xử lý subspace `d_k = d_model / num_heads = 64`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections cho Q, K, V và output
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, _ = x.shape
        # 1. Project x → Q, K, V và split thành nhiều head
        Q = self.W_Q(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Tính scaled dot-product attention song song trên các head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)                                # (B, h, L, d_k)

        # 3. Concat các head → project về d_model
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_O(out)

mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)
y = mha(x)
print(f'Input:  {x.shape}                # (batch, seq_len, d_model)')
print(f'Output: {y.shape}                # cùng shape với input (residual-friendly)')
print(f'Số head: 8 (mỗi head d_k=64)')
print(f'Số tham số MHA: {sum(p.numel() for p in mha.parameters()):,}')

## 2.1.6. Position-wise Feed-Forward Networks

Áp dụng MLP **độc lập trên TỪNG vị trí** (token) trong chuỗi:

$$\text{FFN}(x) = \max(0, xW_1 + b_1) W_2 + b_2$$

Trong T5: dùng ReLU (T5-base) hoặc GeLU (T5-v1.1). `d_ff = 2048` cho T5-base.

In [ ]:
class PositionwiseFFN(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)        # mở rộng 4x
        self.fc2 = nn.Linear(d_ff, d_model)        # thu lại
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # Áp dụng MLP độc lập trên TỪNG token
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

ffn = PositionwiseFFN(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)
y = ffn(x)
print(f'Input:  {x.shape}              # (batch, seq_len, d_model=512)')
print(f'Output: {y.shape}              # cùng shape')
print(f'Hidden d_ff: 2048 (mở rộng 4x rồi thu lại)')
print(f'Số tham số FFN: {sum(p.numel() for p in ffn.parameters()):,}')

## 2.1.7. Positional Encoding

Self-attention **không có** khái niệm thứ tự → cần thêm vector vị trí vào embedding:

$$PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

T5 dùng **Relative Position Encoding** (cải tiến của sinusoidal).

In [ ]:
def positional_encoding(max_len, d_model):
    """Sinusoidal Positional Encoding theo paper Attention Is All You Need."""
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div_term)         # vị trí chẵn dùng sin
    pe[:, 1::2] = torch.cos(pos * div_term)         # vị trí lẻ dùng cos
    return pe

pe = positional_encoding(max_len=100, d_model=64)
print(f'PE shape: {pe.shape}              # (max_len=100, d_model=64)')

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe.numpy(), cmap='RdBu', aspect='auto')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Position in sequence')
ax.set_title('Positional Encoding (sinusoidal)\nPE(pos, 2i) = sin(pos/10000^(2i/d))')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 2.1.8. Residual Connection + Layer Normalization

**Mục đích**: ổn định khi train mạng sâu (T5-base có 24 layers).

$$\text{LayerNorm}(x + \text{Sublayer}(x))$$

T5 đặt LayerNorm **trước** sub-layer (Pre-LN) thay vì sau (Post-LN của Transformer gốc).

In [ ]:
class ResidualLayerNorm(nn.Module):
    """Residual + LayerNorm (Pre-LN style của T5)."""
    def __init__(self, d_model, sublayer):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.sublayer = sublayer  # vd: MultiHeadAttention hoặc FFN

    def forward(self, x):
        # Pre-LN: norm TRƯỚC sub-layer, residual SAU
        return x + self.sublayer(self.norm(x))

x = torch.randn(2, 10, 512)
block = ResidualLayerNorm(d_model=512, sublayer=PositionwiseFFN(512, 2048))
y = block(x)
print(f'Input:  {x.shape}')
print(f'Output: {y.shape}')
print(f'Mean shift trong LayerNorm: {(y - x).mean():.4f}')
print(f'Std shift trong LayerNorm:  {(y - x).std():.4f}')
print('→ Residual giúp gradient flow, LayerNorm giữ scale ổn định')

## 2.2.2. Encoder Block hoàn chỉnh — kết hợp tất cả 5 concept trên

Một Transformer encoder block = **Self-Attention + FFN + Residual + LayerNorm**.

T5-base có **12 encoder blocks** + **12 decoder blocks** = 24 layers.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.mha     = MultiHeadAttention(d_model, num_heads)
        self.ffn     = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Sub-layer 1: Multi-Head Self-Attention với Residual + LayerNorm
        x = x + self.dropout(self.mha(self.norm1(x)))
        # Sub-layer 2: Position-wise FFN với Residual + LayerNorm
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

block = TransformerEncoderBlock(d_model=512, num_heads=8, d_ff=2048)
x = torch.randn(2, 10, 512)
y = block(x)
print(f'Input  shape: {x.shape}')
print(f'Output shape: {y.shape}')
print(f'Tham số 1 block: {sum(p.numel() for p in block.parameters()):,}')
print(f'T5-base có 12 blocks encoder + 12 blocks decoder = 24 layers')

## 2.2.3. Mô hình T5 — load và inspect

T5 (Text-to-Text Transfer Transformer) của Google là **encoder-decoder Transformer** chuyên cho mọi task NLP. Áp dụng cho task của ta:

```
Input:  "fix grammar and spelling: I has went home"
Output: "I have gone home"
```

**Cell sau load T5-base** — để inspect kiến trúc thật.

In [ ]:
!pip install -q transformers sentencepiece

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load T5-base (mô hình ta dùng trong project)
tok = T5Tokenizer.from_pretrained('t5-base')
model = T5ForConditionalGeneration.from_pretrained('t5-base')

# Inspect kiến trúc
print(f'=== T5-base architecture ===')
print(f'Vocab size:      {model.config.vocab_size:,}')
print(f'd_model:         {model.config.d_model}')
print(f'd_ff:            {model.config.d_ff}')
print(f'num_heads:       {model.config.num_heads}')
print(f'num_layers:      {model.config.num_layers} (encoder) + {model.config.num_decoder_layers} (decoder)')
print(f'Total params:    {sum(p.numel() for p in model.parameters()):,}')
print()
print('=== Cấu trúc 1 encoder layer ===')
print(model.encoder.block[0])

In [ ]:
# Demo inference: câu sai → câu đúng
test_inputs = [
    'fix grammar and spelling: I has went to the libary yesterday',
    'fix grammar and spelling: She dont like coffe',
    'fix grammar and spelling: My freind are very kindly person',
]

for inp in test_inputs:
    enc = tok(inp, return_tensors='pt', truncation=True, max_length=128)
    out_ids = model.generate(**enc, max_length=128, num_beams=4)
    out_text = tok.decode(out_ids[0], skip_special_tokens=True)
    print(f'Input:  {inp.replace("fix grammar and spelling: ", "")}')
    print(f'Output: {out_text}\n')

## Tóm tắt mapping concept → code project

| Concept (Chương 2) | Code project | File |
|---|---|---|
| 2.2.1 Seq2Seq | T5ForConditionalGeneration | train_t5_model.ipynb |
| 2.2.2 Transformer | encoder.block + decoder.block | T5 internal |
| 2.1.4 Self-Attention | T5Attention.compute_bias | T5 internal |
| 2.1.5 Multi-Head | num_heads=12 trong T5-base | T5 config |
| 2.1.6 FFN | T5DenseActDense | T5 internal |
| 2.1.7 Positional Enc | T5 dùng RelativePositionBias | T5 internal |
| 2.1.8 Residual+LayerNorm | T5LayerNorm + skip connections | T5 internal |
| 2.2.3 T5 | t5-base from HuggingFace | train_t5_model.ipynb |

## Đoạn báo cáo Chương 5 (Cài đặt)

> *Hệ thống Spell-Checker sử dụng mô hình **T5-base** (Text-to-Text Transfer Transformer, Raffel et al. 2020) — kiến trúc Encoder-Decoder Transformer với 222.9 triệu tham số, gồm 12 encoder blocks và 12 decoder blocks. Mỗi block bao gồm: (1) **Multi-Head Self-Attention** với 12 heads (d_k=64) áp dụng công thức Scaled Dot-Product `softmax(QKᵀ/√d_k)V`; (2) **Position-wise Feed-Forward Network** với d_ff=3072; (3) **Residual Connection + Layer Normalization** theo Pre-LN style. T5 sử dụng **Relative Position Bias** thay cho Sinusoidal Positional Encoding gốc, giúp generalize tốt hơn với chuỗi dài. Mô hình được fine-tune trên dataset 26,856 cặp (sai, đúng) tổng hợp từ 6 nguồn, đạt BLEU=66.97, GLEU=0.6698, F0.5=0.9627 trên test set 2,686 câu.*